In [1]:
from dotenv import load_dotenv
import os

In [2]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()
instructions = parser.get_format_instructions()
print(instructions) 

Return a JSON object.


In [3]:
ai_response = '{"이름": "나대로", "나이": 30}'
parsed_response = parser.parse(ai_response)
print(parsed_response)

{'이름': '나대로', '나이': 30}


In [4]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "너는 {story}에 나오는 {character_a} 역할이다. 그 캐릭터에 맞게 사용자와 대화하라."
human_template = "안녕? 저는 {character_b}입니다. 오늘 시간 괜찮으시면 {activity}까요?"

prompt_template = ChatPromptTemplate([
    ("system", system_template),
    ("user", human_template),
])

result = prompt_template.invoke({
    "story": "신데렐라",
    "character_a": "왕자",
    "character_b": "신데렐라",
    "activity": "춤을 출"
})

print(result)

messages=[SystemMessage(content='너는 신데렐라에 나오는 왕자 역할이다. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕? 저는 신데렐라입니다. 오늘 시간 괜찮으시면 춤을 출까요?', additional_kwargs={}, response_metadata={})]


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
model = ChatOpenAI(model="gpt-4o-mini")

chain = prompt_template | model | parser

chain.invoke({
    "story": "신데렐라",
    "character_a": "왕자",
    "character_b": "신데렐라",
    "activity": "춤을 출"
})

'안녕하세요, 아름다운 신데렐라. 당신과 함께 춤추는 것이라면 어떤 기쁜 일도 마다하지 않겠습니다. 우리의 발걸음이 하나가 되는 순간을 함께 즐깁시다. 어떤 음악을 좋아하시나요?'

In [10]:
chain.invoke({
    "story": "춘향전",
    "character_a": "성춘향",
    "character_b": "변사또",
    "activity": "내 수청을 들"
})

'안녕하세요, 변사또님. 저는 춘향입니다. 저에게 수청을 요구하신다면 정말 어이가 없습니다. 제 마음은 이미 이도령께 하고 있으니, 그 요청을 정중히 거절하겠습니다. 제 마음과 뜻을 헤아려 주시기 바랍니다.'

In [11]:
from typing import Literal
from pydantic import BaseModel, Field

class Scenario(BaseModel):
    """스토리 설정과 사용자 입력에 반응하는 대사를 만드는 클래스"""
    answer: str = Field(description="스토리 설정과 사용자와의 대화 기록에 따라 생성된 대사")
    emotion: Literal["기쁨", "분노", "슬픔", "공포", "냉소", "불쾌", "중립"] = Field(description="대사의 주요 감정")
    main_emotion_intensity: float = Field(description="대사의 주요 감정의 강도 (0.0 ~ 1.0)")

structured_llm = model.with_structured_output(Scenario)
scenario_chain = prompt_template | structured_llm

scenario_chain.invoke({
    "story": "신데렐라",
    "character_a": "왕자",
    "character_b": "신데렐라",
    "activity": "춤을 출"
})


Scenario(answer='안녕하세요, 아름다운 신데렐라! 당신과 함께 춤을 출 수 있다니, 정말 영광입니다. 당신의 발걸음에 맞춰, 저와 함께 아름다운 순간을 만들어봅시다.', emotion='기쁨', main_emotion_intensity=0.9)

In [12]:
scenario_chain.invoke({
    "story": "춘향전",
    "character_a": "성춘향",
    "character_b": "변사또",
    "activity": "내 수청을 들"
})

Scenario(answer='변사또님, 저와의 약속은 지켜주셔야죠. 저는 마음속에 정한 사랑이 있으니 그 마음 변치 않을 것입니다. 저를 잘 설득하시려면 그에 맞는 진정한 마음을 보여주셔야 할 것입니다.', emotion='분노', main_emotion_intensity=0.8)